In [1]:
#Librerias utilizadas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import time
import math
import glob
from pathlib import Path

#Global variables to table format in latex
TAB_LINKER = " & "
TAB_END = "\\\\"
CHECK_SYMBOL = "\\checkmark"
CROSS_SYMBOL = "\\texttimes"

In [2]:
def get_ListOfDF(directory:str, verbose:int = 0):
    # Read all the results tables in the directory
    csv_files = glob.glob(directory)

    # Sort the tables by name
    csv_files = sorted(csv_files)

    # Read all the csv files from the directory, and save we save them in a list
    dataframes = [pd.read_csv(f) for f in csv_files]
    names_frames = [Path(csv).stem.split('_')[0] for csv in csv_files]

    #We print the name of the readed tables if it's requiered
    if verbose in [-1, 0]: print(names_frames)
    if verbose in [0,1]: print("Readed path: ", directory)

    # We print the tables readed
    if verbose == 0:
        for i in range(0, len(csv_files)):
            print(f"|{i}| The table has been read it from the directory: {csv_files[i]}")
        print("-"*80)
    if verbose in [-1,0,1]:
        print(f"{len(dataframes)} tables has been readed.")

    return dataframes, names_frames

#Directory of the results original
directory_original = "./csvs/results/original/*.csv"
directory_addedLS = "./csvs/results/addedLS/*.csv"
directory_simplest = "./csvs/results/simplest/*.csv"

#Read all the results of the dataframes
dataframes_orig, names_frames = get_ListOfDF(directory_original, -1)
dataframes_LS, names_frames = get_ListOfDF(directory_addedLS, -1)
dataframes_SP, names_frames = get_ListOfDF(directory_simplest, -1)

['ARGLINA', 'BARD', 'BEALE', 'BRKMCC', 'BROWNAL', 'BROWNBS', 'BROWNDEN', 'CHNROSNB', 'CLIFF', 'CUBE', 'DECONVU', 'DENSCHNA', 'DENSCHNC', 'DENSCHND', 'DENSCHNF', 'DIXON3DQ', 'EIGENALS', 'EIGENBLS', 'ENGVAL2', 'FLETCBV2', 'FLETCHCR', 'GENHUMPS', 'HAIRY', 'HEART6LS', 'HELIX', 'HILBERTA', 'HILBERTB', 'HIMMELBB', 'HIMMELBH', 'HUMPS', 'JENSMP', 'KOWOSB', 'LOGHAIRY', 'MANCINO', 'MARATOSB', 'MEXHAT', 'PALMER1C', 'PALMER2C', 'PALMER3C', 'PALMER4C', 'PALMER5C', 'PALMER6C', 'PALMER7C', 'PALMER8C', 'ROSENBR', 'SINEVAL', 'SISSER', 'TOINTQOR', 'VARDIM', 'WATSON', 'YFITU']
51 tables has been readed.
['ARGLINA', 'BARD', 'BEALE', 'BRKMCC', 'BROWNAL', 'BROWNBS', 'BROWNDEN', 'CHNROSNB', 'CLIFF', 'CUBE', 'DECONVU', 'DENSCHNA', 'DENSCHNC', 'DENSCHND', 'DENSCHNF', 'DIXON3DQ', 'EIGENALS', 'EIGENBLS', 'ENGVAL2', 'FLETCBV2', 'FLETCHCR', 'GENHUMPS', 'HAIRY', 'HEART6LS', 'HELIX', 'HILBERTA', 'HILBERTB', 'HIMMELBB', 'HIMMELBH', 'HUMPS', 'JENSMP', 'KOWOSB', 'LOGHAIRY', 'MANCINO', 'MARATOSB', 'MEXHAT', 'PALMER1C', 

In [ ]:
# Dictionary of the dimensions of the testet problems
problem_dimensions = {
    "ARGLINA": 200,
    "BARD": 3,
    "BEALE" : 2,
    "BRKMCC": 2,
    "BROWNAL": 200,
    "BROWNBS": 2,
    "BROWNDEN": 4,
    "CHNROSNB": 50,
    "CLIFF": 2,
    "CUBE": 2,
    "DECONVU": 63,
    "DENSCHNA": 2,
    "DENSCHNB": 2,
    "DENSCHNC": 2,
    "DENSCHND": 2,
    "DENSCHNF": 2,
    "DIXON3DQ": 10000,
    "EIGENALS": 2550,
    "EIGENBLS": 2550,
    "ENGVAL2": 3,
    "EXTROSNB": 1000,
    "FLETCBV2": 5000,
    "FLETCHCR": 1000,
    "GENHUMPS": 5000,
    "HAIRY": 2,
    "HEART6LS": 6,
    "HELIX": 3,
    "HILBERTA": 2,
    "HILBERTB": 10,
    "HIMMELBB": 2,
    "HIMMELBH": 2,
    "HUMPS": 2,
    "JENSMP": 2,
    "KOWOSB": 4,
    "LOGHAIRY": 2,
    "MANCINO": 100,
    "MARATOSB": 2,
    "MEXHAT": 2,
    "PALMER1C": 8,
    "PALMER2C": 8,
    "PALMER3C": 8,
    "PALMER4C": 8,
    "PALMER5C": 5, 
    "PALMER6C": 8,
    "PALMER7C": 8,
    "PALMER8C": 8,
    "ROSENBR": 2,
    "SINEVAL": 2,
    "SISSER": 2,
    "TOINTQOR": 50,
    "VARDIM": 200,
    "WATSON": 12,
    "YFITU": 3
}

In [5]:
def construct_matrixResults(list_dataframes: list[pd.DataFrame])->np.ndarray:
    """
    # Usage
    Function to get the values from the dataframes. This function create a Numpy
    array of 3 dimensions, the measured variable (first dimension), the number of 
    problem, and the method. 

    ## Input. 
        - ```list_dataframes```: list[pd.DataFrame] - 'Descriptive name'

    ## Output.
        - ```result_tensor```: np.ndarray - Tensor containing all the results.

    The order of the variables is the same (check remarks for more information),
    
    ## Remarks.
    The measured variables were (in the given order in the csv files):

        1 - Iterations. The number of iterations that the algorithm took
        2 - Execution time. Time that the algorithm took.
        3 - LastNorm. The magnitud of the gradient from the last element in the sequence.
        4 - Iterations per Second - Iterations/Execution_time
        5 - Convergence. If the algorithm converged in the given iterations and minimal 
        acceptable gradient.

    The problem solved are described in the ```problem_dimension``` variable. And the order
    of the problems are

        0 - AMG. NAMGM/Accelerated minimal gradient set of vectors
        1 - Grads. NAMGM/Gradient set of vectors
        2 - Random. NAMGM/Random set of vectors
        3 - Newton. ¨¨
        4 - BFGS. ¨¨
        5 - GDLS. Gradient descent with Line Search (this if fix for all problems)          

    ## Example usage.
    To observe the resutls of number of iterationsuse ```result_tensor[0, :, :]```. 
    Then the row dimension are the problems and the columns are the different methods.
    """
    # We have 5 methods and 5 variables, then in total we have N \times 5 elements in the dataframe
    result_tensor = np.zeros((5, len(list_dataframes), 6)) #(Mesuared Variables, Problems, Methods)

    #Recolection of the information of each method (recolected in rows)
    for i in range(0, len(list_dataframes)):
        convergence = list_dataframes[i]["Archived Convergence"].to_numpy()
        itterations = list_dataframes[i]["iterations"].to_numpy()
        grads = list_dataframes[i]["Last Gradient"].to_numpy()
        exc = list_dataframes[i]["Execution time"].to_numpy()
        ittPSec = list_dataframes[i]["Iterations per Second"].to_numpy()

        #Setting the information in the result matrix
        result_tensor[:, i, :] = [itterations, exc, grads, ittPSec, convergence]

    return result_tensor

#Names of the methods
col_names = ["GDLS", "Oviedo", "Grads", "Random",  "BFGS", "Newton"]
column_names_addedLS = ["G", "A", "Q", "R", "B", "N"]

#Order of the variables:
variable_names = ("iterations", "Last gradient", "Execution time", "Iterations per Second", "Archived Convergence")

#Get the matrix of results
result_tensor_original = construct_matrixResults(dataframes_orig)
result_tensor_addedLS = construct_matrixResults(dataframes_LS)
result_tensor_simplest = construct_matrixResults(dataframes_SP)

def change_orderOfMethods(result_tensor:np.ndarray, neworder:list[int] = [0, 1, 2, 3, 4, 5]):
    """
    # Usage
    Function to modify the order of the presented method in the last dimension.
    The original order of the methods is.
    
    - 0 - NAMGM/Using the AMGM set of vectors
    - 1 - NAMGM/Using the Queue set of vectors
    - 2 - NAMGM/Using the Random set of vectors
    - 3 - Modified Newton's method
    - 4 - BFGS method
    - 5 - Gradient Descent with Line Search
    
    ## Input.
        - ```result_tensor```: np.ndarray - 'Descriptive name'
        - ```neworder```: list[int] - New proposed order

    ## Output.
        - ```result_tensor```: np.ndarray - Same tensor results but a new method otder    
        
    ## Remark. 
    The use of line search (LS) does not modify the order, this is the given order 
    for all the different kind of experiments with or without the use of LS.
    """

    assert len(neworder) == result_tensor.shape[2], "The given reorder does not match with the dimension of the methods"
    reOrdered_matrix = result_tensor[:, :, neworder]
    return reOrdered_matrix

#Reorder the matrix results
result_tensor_original = change_orderOfMethods(result_tensor_original, [5, 0, 1, 2, 4, 3])
result_tensor_addedLS = change_orderOfMethods(result_tensor_addedLS, [5, 0, 1, 2, 4, 3])
result_tensor_simplest = change_orderOfMethods(result_tensor_simplest, [5, 0, 1, 2, 4, 3])

In [6]:

#Configuration of the printing 
default_values = [1000, 100, -np.inf, np.inf, np.inf]

def isRepeteatedValue(value, array:np.ndarray):
    counter = np.sum(array == value)
    if counter > 1: return True
    else: return False

def highlight_result(val:any, math_mode:bool)->str:
    "Highlight the result for a value. There are two modes, normal and math "    
    return f"$\\bs{{{val}}}$" if math_mode else f"\\textbf{{{val}}}"

def rewrite_value_in_str(val:any, isInteger:bool = False)->str:
    """Equivalent to \\tableHL just using other alternvative to save space"""
    # Get the exponent and the decimal from the value
    exponent = int(math.floor(math.log10(abs(val))))
    val_str = f"{val :.3e}"
    if not isInteger:
        if -6 < exponent <= 4:
            content: str = f"$\\mathbf{{{val:.3f}}}$"
        else:
            content: str = "$\\mathbf{" +val_str[0:5] + "\\times10^{" + str(int(val_str[-3:])) + "}" +"}$" 
        return content
    else:
        val =int(val)
        return f"$\\mathbf{{{val:d}}}$"
    
def construct_cell_color(color1:str, alpha1:int, color2:str =None, alpha2:int=None)->str:
    """# Usage
    
    Function to construct the color cell on a LaTeX table. If color2 and alpha 2 are
    not provided, it still works.
    
    ## Input:
        - ``color1``: string - First (or main) color of the cell
        - ``alpha1``:   int  - Transparency (or percentage of the first color) 
        - ``color1``(optional): string - Second color of the cell
        - ``alpha2``(optional):   int  - Transparency of the combined color 

    ## Output:
        - ``s``: string - LaTeX command to color the cell
    """
    final_color:str = f"{color1}!{alpha1}"
    if color2:
        final_color += f"!{color2}"
        if alpha2:
            final_color += f"!{alpha2}"
    return f"\\cellcolor{{{final_color}}}"
 

def printRAWInfo(result_tensor: np.ndarray, columnToPrint:int, addProblemName:bool = False, addProblemDim: bool = False, 
                 showInfo:bool = False, dimBoundLower:int = 0, dimBoundUpper:int = np.inf, add_convergence_color:bool = False,
                 alpha_cell:int = 10):
    """"""
    #Assert that the column to be printed is valid
    assert columnToPrint in [0, 1, 2, 3], "Not valid entry to print in the result matrix"
    
    
    #Print the information
    printed_values = 0 
    max_intValue = default_values[columnToPrint]

    #Move over the problems (rows)
    for (i, problem) in enumerate(names_frames):
        problemDim = problem_dimensions[problem] #Dimension problem
        #If the dimension is on the given interval, then print
        if dimBoundLower <= problemDim <= dimBoundUpper: 
            s:str = ""

            #Add the problem name if it's requiered
            if addProblemName: s += problem + TAB_LINKER

            #Add the dimension of the problem if it's requiered
            if addProblemDim and addProblemName: s += f"{str(problemDim)}" + TAB_LINKER


            #Obtain the index of the winner value
            best_value_index = result_tensor[columnToPrint, i, :].argmin() if columnToPrint in [0, 1, 2] else result_tensor[columnToPrint, i, :].argmax()
            
            #Then for all result in the set of methods (all the problems)
            for j in range(6):

                #Obtain the value
                value = result_tensor[columnToPrint, i, j]

                if add_convergence_color: 
                    #If the method converged or not, we color the cell (any symbol added)
                    s += construct_cell_color("ForestGreen", alpha_cell) if bool(result_tensor[-1, i, j]) else construct_cell_color("gray", alpha_cell)
                

                #Get the best value in the result
                best_value = result_tensor[columnToPrint, i, best_value_index]

                #And verify that it's unique
                repetition_of_values = isRepeteatedValue(best_value, result_tensor[columnToPrint, i, :])

                if value == 0.0: 
                    if value == best_value and value != max_intValue and repetition_of_values:
                       s += "{{$\\bs{{{}}}$}}".format(int(value))
                    else: s += "{0}"
                elif value == np.inf: s += f"\cellcolor{{red!20}}{{{CROSS_SYMBOL}}}"
                
                else:    
                    #If the value is the winer then, highlight it
                    if value == best_value and not repetition_of_values:
                        #If it's the iteration column
                        s += rewrite_value_in_str(value, isInteger=True) if (columnToPrint==0 and j!=3) else rewrite_value_in_str(value)  
                    else:
                        if value in default_values:
                            s += "M"
                        else:
                            #If it's the iteration column
                            s += str(int(value)) if (columnToPrint==0) else str(value)
                #Separator of columns and rows
                if j != 5: s += TAB_LINKER
                else: s += TAB_END
            print(s)
            printed_values += 1 
        
        #Otherwise don't print the row
        else: pass   
    #Show the information of the printed table
    if showInfo:
        print("Table variable being written:", variable_names[columnToPrint])
        print(f"In total were printed {printed_values} rows.")


In [7]:

#First column of results (Number of iterations)
#Lowdimensional problems
printRAWInfo(result_tensor_simplest, columnToPrint=0, addProblemName = False, addProblemDim=False, 
            showInfo=False, dimBoundUpper = 1999, add_convergence_color=True) 
printRAWInfo(result_tensor_simplest, 0, addProblemName = False, addProblemDim=False, showInfo=False, 
            dimBoundLower = 2000, add_convergence_color=True) #Highdimensional problems

#Second column of results (Last gradient)
#printRAWInfo(result_tensor_original, 1, addProblemName = False, addProblemDim=False, showInfo=False, dimBoundLower=2000)
#printRAWInfo(2, addProblemName = False, addProblemDim=False, showInfo=False, dimBoundUpper=1999)

#Third column of results (Execution time)
#printRAWInfo(3, addProblemName = True, addProblemDim=False, result_tensor=False, dimBoundUpper = 1999)
#printRAWInfo(3, addProblemName = True, addProblemDim=False, showInfo=False, dimBoundLower = 2000)

#Last column of results (Iterations per second)
#printRAWInfo(4, addProblemName = False, addProblemDim=False, result_tensor=False, dimBoundUpper = 1999)
#printRAWInfo(4, addProblemName = False, addProblemDim=False, result_tensor=False, dimBoundLower= 2000)

\cellcolor{ForestGreen!10}2 & \cellcolor{ForestGreen!10}2 & \cellcolor{ForestGreen!10}2 & \cellcolor{ForestGreen!10}2 & \cellcolor{ForestGreen!10}2 & \cellcolor{ForestGreen!10}2\\
\cellcolor{gray!10}M & \cellcolor{ForestGreen!10}64 & \cellcolor{ForestGreen!10}17 & \cellcolor{ForestGreen!10}9 & \cellcolor{ForestGreen!10}32 & \cellcolor{ForestGreen!10}9\\
\cellcolor{gray!10}M & \cellcolor{ForestGreen!10}33 & \cellcolor{ForestGreen!10}59 & \cellcolor{ForestGreen!10}2 & \cellcolor{ForestGreen!10}2 & \cellcolor{ForestGreen!10}2\\
\cellcolor{gray!10}M & \cellcolor{ForestGreen!10}5 & \cellcolor{ForestGreen!10}5 & \cellcolor{ForestGreen!10}4 & \cellcolor{ForestGreen!10}5 & \cellcolor{ForestGreen!10}4\\
\cellcolor{gray!10}M & \cellcolor{ForestGreen!10}$\mathbf{4}$ & \cellcolor{ForestGreen!10}6 & \cellcolor{gray!10}M & \cellcolor{gray!10}M & \cellcolor{ForestGreen!10}15\\
\cellcolor{gray!10}M & \cellcolor{ForestGreen!10}69 & \cellcolor{ForestGreen!10}42 & \cellcolor{ForestGreen!10}$\mathbf{9.933

In [8]:
#print("Number of problems:", len(names_frames))

def generate_stateTable(result_matrix, names_frames, dimensions:dict, 
                        dimBoundLower: int = 0, dimBoundUpper:int = np.inf, show_dimension:bool = True, 
                        colorBoundLower:str = "ForestGreen", colorBoundUpper:str = "FireRed", alpha=0.1,
                        separation:int = 0, only_show_Randoms:bool = True, show_probs: bool = False, use_symbols:bool=True,
                        colorSymbols:bool = True, colorSymbolCheck:str = "ForestGreen", colorSymbolCross:str = "Red",
                        printfinalTable:bool = False):
    setOfRows = []
    rows_printed = []
    elementsPrinted = 0
    # Generate the body of the convergence table.
    for (i, problems_name) in enumerate(names_frames):
        
        #Get the dimension of the problem
        dimproblem = dimensions[problems_name]

        #If the problem satisfy the minimum dimension then it's printed
        if dimBoundLower <= dimproblem <= dimBoundUpper:
            s = problems_name + f" & "
            if show_dimension: s+= f"{dimproblem} &"
            for j in range(0, 6):

                #Get the value of the matrix
                value = result_matrix[0, i, j]
                
                #Add the color depending the percentage of convergence
                percentage = int(value * 100)
                s += "\cellcolor{"+colorBoundLower+f"!{percentage}!"+colorBoundUpper+f"!{int(alpha*100)}"+"}"

                #Add the probability whether it's number or symbol
                if show_probs:
                    if only_show_Randoms:
                        if value!= 0.0 and value!=1.0:
                            s += f" {{\\scriptsize {value:.1f}}}"
                    if use_symbols:
                        if not colorSymbols:
                            if value == 0.0:s += "\\texttimes"
                            elif value == 1.0:s += "\\checkmark"
                        else:
                            if value == 0.0:s += f"\\textcolor{{{colorSymbolCross}}}"+"{\\texttimes}"
                            elif value == 1.0:s += f"\\textcolor{{{colorSymbolCheck}}}"+"{\\checkmark}"
                    else:
                        s += f" {{\\scriptsize {value:.1f}}}"
                
                #Separators of colums
                if j != 5: s += " & "
                else: s += " \\\\"

            #Print the constructed row of the table
            elementsPrinted+=1
            rows_printed.append(s + "\n")
            #print(s)
            
            #To break the table into other subtable
            if separation!=0:
                if ((elementsPrinted % separation)==0):
                    if printfinalTable: 
                        print("".join(rows_printed))
                        print("------ Table Break ------")                        
                    setOfRows.append(rows_printed)
                    rows_printed = []
        else:
            pass
    if printfinalTable: print("".join(rows_printed))
    setOfRows.append(rows_printed)
    print(f"{elementsPrinted} lines has been printed.")
    return setOfRows


def printTable(rows:list[str], columns_name:list[str], style="c"):
    """Print the table using the given separation"""
    
    for setOfRows in rows:
        #HEARDER OF THE TABLE
        table_command = "\\begin{tabular}"+"{"+ style+ "}\n"
        table_command += "\\toprule\n"

        #NAME OF THE COLUMNS
        columns_rows = [columns + " & " if i != len(columns_name)-1 else columns + "\\\\ \n" for i, columns in enumerate(columns_name)]
        table_command += "".join(columns_rows)
        table_command += "\\midrule\n"
        
        #Body of the table
        table_command += "".join(setOfRows)

        #Final of the table
        table_command += "\\bottomrule\n"
        table_command += "\\end{tabular}"

        print(table_command)    
        yield


def printHoleTable(tbodys:list[str], columns_name:list[str], subtables_nummer:int,
                style:str="c", position:str = "H", addCentering:bool = True, scaleboxfactor:float = 1.0):
    
    #Assertion of scaleboxfactor
    assert 0.0 < scaleboxfactor <= 1, "Not acceptable value for scalebox, must be on interval (0,1]"
    
    #Enviroment of the tables
    print("\\begin{table}"+f"[{position}]")
    if addCentering: print("\\centering")
    print(f"\\scalebox{{{scaleboxfactor:.2f}}}"+"{")
    print("\\begin{tabular}"+"{"+ "c"*subtables_nummer + "}")

    tables = printTable(tbodys, columns_name, style)
    numberOfTables = len(tbodys)
    for i in range(1,numberOfTables+1):
        next(tables)
        if i % subtables_nummer == 0: print("\\\\")
        else: print(" & ")
    print("\\end{tabular}}")
    print("\\end{table}")

#Generate table (All problems without probs)
# tables = generate_stateTable(result_matrix, names_frames, problem_dimensions, separation= 9, show_dimension=False, 
#                     colorBoundUpper="gray", colorBoundLower="ForestGreen", alpha=0.25, 
#                     show_probs=True, use_symbols=True, colorSymbolCross="black", printfinalTable=True)

# #Generate table (Low dimensional problems 0-1999)
# tables = generate_stateTable(result_matrix, names_frames, problem_dimensions, show_dimension=False, dimBoundUpper=1999, separation=10, 
#                     colorBoundUpper="gray", colorBoundLower="ForestGreen", alpha=0.25, 
#                     show_probs=True, use_symbols=True, colorSymbolCross="black", printfinalTable=True)

#Generate table (High dimensional problems 2000-\inf)
table =  generate_stateTable(result_matrix, names_frames, problem_dimensions, show_dimension = False, dimBoundLower=2000, separation=0, 
                    colorBoundUpper="gray", colorBoundLower="ForestGreen", alpha=0.25, 
                    show_probs=True, use_symbols=True, colorSymbolCross="black", printfinalTable=True)


NameError: name 'result_matrix' is not defined

In [ ]:
#Print the table for the added LS results
printHoleTable(table, ["Fname"]+column_names_addedLS, 3, style="l"+"|c"*6+"|", scaleboxfactor=0.6)

#Print the compelete table 
#printHoleTable(tables, ["Fname"]+col_names_LS, 3, style="l"+"|c"*6+"|", scaleboxfactor=0.6)

\begin{table}[H]
\centering
\scalebox{0.60}{
\begin{tabular}{ccc}
\begin{tabular}{l|c|c|c|c|c|c|}
\toprule
Fname & G & A & Q & R & B & N\\ 
\midrule
DIXON3DQ & \cellcolor{ForestGreen!10000!gray!25} {\scriptsize 100.0} & \cellcolor{ForestGreen!10000!gray!25} {\scriptsize 100.0} & \cellcolor{ForestGreen!10000!gray!25} {\scriptsize 100.0} & \cellcolor{ForestGreen!10000!gray!25} {\scriptsize 100.0} & \cellcolor{ForestGreen!200!gray!25} {\scriptsize 2.0} & \cellcolor{ForestGreen!200!gray!25} {\scriptsize 2.0} \\
EIGENALS & \cellcolor{ForestGreen!10000!gray!25} {\scriptsize 100.0} & \cellcolor{ForestGreen!10000!gray!25} {\scriptsize 100.0} & \cellcolor{ForestGreen!10000!gray!25} {\scriptsize 100.0} & \cellcolor{ForestGreen!10000!gray!25} {\scriptsize 100.0} & \cellcolor{ForestGreen!10000!gray!25} {\scriptsize 100.0} & \cellcolor{ForestGreen!10000!gray!25} {\scriptsize 100.0} \\
EIGENBLS & \cellcolor{ForestGreen!10000!gray!25} {\scriptsize 100.0} & \cellcolor{ForestGreen!10000!gray!25} {\scri

# Creation of a summary table


In [ ]:
def generate_SummaryFloatVal(name_frmaes, cellcolors:list[str], max_default_values:list[any], verbose:int = 0):

    #Initial values
    indices_values = [[], [], [], [], []]
    printed_values = 0

    #Move throught the problem name
    for (i, problem) in enumerate(name_frmaes):
        s = problem + " & "
        #Move throught all the 5 variables (convergence don't is being used)
        for w in range(1, 5):
            if w in [1, 2, 3]:
                best_value_index = result_matrix[w, i, :].argmin()
            else:
                best_value_index = result_matrix[w, i, :].argmax()
            if w != 4:
                s += cellcolors[best_value_index] + " & "
            else:
                s += cellcolors[best_value_index]
            if result_matrix[w, i, best_value_index] != max_default_values[w]:
                indices_values[w].append(int(best_value_index))
        s += "\\\\"
        print(s)
        printed_values += 1
    
    #Information
    if verbose == 0: print("Total lines printed", printed_values)


#Printed values (it should follow the given order in the result matrix)
cellcolors = ["\\cellcolor{blue!20}A", "\\cellcolor{red!20}G", "\\cellcolor{purple!20}R", "\\cellcolor{green!20}N", "\\cellcolor{yellow!20}B"]
cellcolors_LS =  ["\\cellcolor{orange!20}S", "\\cellcolor{blue!20}A", "\\cellcolor{red!20}Q", "\\cellcolor{purple!20}R", "\\cellcolor{yellow!20}B", "\\cellcolor{green!20}N"]
default_values = [None, 1000, 0, np.inf, np.inf, np.inf, ]

#Print the table
generate_SummaryFloatVal(names_frames, cellcolors_LS, default_values, 0)
    

ARGLINA & \cellcolor{orange!20}S & \cellcolor{orange!20}S & \cellcolor{blue!20}A & \cellcolor{orange!20}S\\
BARD & \cellcolor{blue!20}A & \cellcolor{green!20}N & \cellcolor{blue!20}A & \cellcolor{blue!20}A\\
BEALE & \cellcolor{yellow!20}B & \cellcolor{green!20}N & \cellcolor{red!20}Q & \cellcolor{yellow!20}B\\
BRKMCC & \cellcolor{blue!20}A & \cellcolor{purple!20}R & \cellcolor{blue!20}A & \cellcolor{blue!20}A\\
BROWNAL & \cellcolor{red!20}Q & \cellcolor{green!20}N & \cellcolor{blue!20}A & \cellcolor{blue!20}A\\
BROWNBS & \cellcolor{blue!20}A & \cellcolor{purple!20}R & \cellcolor{blue!20}A & \cellcolor{blue!20}A\\
BROWNDEN & \cellcolor{green!20}N & \cellcolor{green!20}N & \cellcolor{blue!20}A & \cellcolor{blue!20}A\\
CHNROSNB & \cellcolor{green!20}N & \cellcolor{green!20}N & \cellcolor{blue!20}A & \cellcolor{blue!20}A\\
CLIFF & \cellcolor{red!20}Q & \cellcolor{purple!20}R & \cellcolor{blue!20}A & \cellcolor{blue!20}A\\
CUBE & \cellcolor{yellow!20}B & \cellcolor{yellow!20}B & \cellcolor{

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 2. Map values to your specific names
# Replace 'Name A', etc., with your actual category names
sns.set_theme(style="whitegrid")
name_map = {
    0: 'Oviedo',
    1: 'Gradients',
    2: 'Random',
    3: 'Newton',
    4: 'BFGS'
}
varibles = ["Iterations", "Last Gradient", "Execution time", "Iterations per second"]


# 3. Convert to Long-Format DataFrame and apply the names
data_list = []
for i, sublist in enumerate(indices_values[1:]):
    for value in sublist:
        data_list.append({
            'Element': varibles[i], 
            'Label': name_map[value]  # Use the name instead of the number
        })
df = pd.DataFrame(data_list)

# 4. Update the color dictionary to use the new names
color_dict = {
    'Oviedo': 'blue',
    'Gradients': 'red',
    'Random': 'purple',
    'Newton': 'green',
    'BFGS': 'orange'
}

# 5. Plotting
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

ax = sns.countplot(
    data=df, 
    x='Element', 
    hue='Label', 
    palette=color_dict,
    hue_order=list(name_map.values()) # Keeps categories in order A -> E
)

plt.title('Frequency per Position with Custom Category Names', fontdict={"fontsize":12}, fontweight='bold', loc="left")
plt.xlabel('Measuraments', fontdict={"fontsize":8})
plt.ylabel('')
plt.grid(alpha=0.8)
plt.legend(title='Methods', loc='upper left')
plt.tight_layout()

#Save plot (For Poster)
#plt.savefig("images/countplot_solvedProblems.jpg", dpi=1200)

#Save plot (For thesis)
#plt.savefig("images/countplot_solvedProblems.svg")


NameError: name 'indices_values' is not defined

## Result concept.
There are so many results. Therefore, I honestly think that is posibily to show to different results of the algorithms for funtion.
Where each row are the results for different Hessian modification. and the columns are measured variables. For example.


**[REMARK]** The following cells only contain the tools to create the table. To observe the creation check the next cell.

In [42]:

def get_scientific_notation(value:float, marker:str = "\\cdot", add_mathmode:bool = True)->float:
    #Frontier case were the value is identically zero
    if value == 0.0:
        return "0"
    
    #Other wise we can obtanin the exponent and 
    else:
        # Get the exponent and the decimal from the value
        exponent = int(math.floor(math.log10(abs(value))))
        decimal = value / (10 ** exponent)

        #Construct the element in the table
        value_scientific_notation = f"{decimal:.3f} " + marker + " 10^{" + str(int(exponent)) +"}"
        value_scientific_notation = f"${value_scientific_notation}$" if add_mathmode else value_scientific_notation
    return value_scientific_notation

def add_bold_to_element(element:str, math_mode:bool, math_bold_command:str = '\\mathbf',
                        is_best_value:bool = False, color_best:str = 'ForestGreen', add_mathmode:bool = True,
                        it_converged: bool = True)->str:
    
    #Variable where we will put the bolded string
    bolded_element: str = ""

    if it_converged:
        if math_mode:
            bolded_element = math_bold_command + f"{{{element}}}" 
            if add_mathmode:
                bolded_element = f"${bolded_element}$"
        else:
            bolded_element = f"\\textbf{{{element}}}"
            
        bolded_element = f"\\textcolor{{{color_best}}}{{{bolded_element}}}" if is_best_value else bolded_element            
        return bolded_element
    else:
        return element
    


#Function to print the results depending on the value on the column
def format_column(value:any, method_int:int, variable_int: int, 
                  use_symbols:bool, use_colors:bool, 
                  final_element:bool = False,
                  is_local_best:bool = False, is_global_best:bool = False,
                  it_converged:bool = True)->str:
    """
    Function to print the results depending on the value on the column 
    """ 

    #String where save the element of the table
    s: str = ""   

    if np.isinf(value):
        s += f"\\textcolor{{BrickRed}}{{{CROSS_SYMBOL}}}"
        #s += construct_cell_color("BrickRed", 10) + CROSS_SYMBOL
    #If the value is the first the number of iterations and it's not 
    # the Random set of vectors, then we print as integer, otherwise
    # we verify that if the value is an integer.
    elif variable_int == 0:
        
        if value.is_integer():
            value = int(value)
            s += f"{value: d}"
        else:
            s += f"{value: .3f}"

        if is_local_best and is_global_best:
            s = add_bold_to_element(s, False, is_best_value=True, it_converged=it_converged)
        elif is_local_best:
            s = add_bold_to_element(s, False, is_best_value=False, it_converged=it_converged)
        else: pass



    #Time variable
    elif variable_int == 1: 
        s += f"{value: .4g}"
        if is_local_best and is_global_best:
            s = add_bold_to_element(s, False, is_best_value=True, it_converged=it_converged)
        elif is_local_best:
            s = add_bold_to_element(s, False, is_best_value=False, it_converged=it_converged)
        else: pass
        


    #Last gradient variable
    elif variable_int == 2:
        #Normally this values are small enought to use scientific notation
        #The refore we will expote this feature. First using the value
        #we obtain the mantisa and exponent
        
        if is_local_best and is_global_best:
            s +=  get_scientific_notation(value, add_mathmode=False)
            s = add_bold_to_element(s, True, is_best_value=True, add_mathmode=True, it_converged=it_converged)
        elif is_local_best:
            s +=  get_scientific_notation(value, add_mathmode=False)
            s = add_bold_to_element(s, True, is_best_value=False, add_mathmode=True, it_converged=it_converged)
        else: 
            s +=  get_scientific_notation(value)

    
    #Iterations per second
    elif variable_int == 3: 
        s += f"{value: .2f}"
        if is_local_best and is_global_best:
            s = add_bold_to_element(s, False, is_best_value=True, it_converged=it_converged)
        elif is_local_best:
            s = add_bold_to_element(s, False, is_best_value=False, it_converged=it_converged)
        else: pass

    #Convergence flag 
    else:
        convergence_flag = bool(value)
        if value != 0.0 and value != 1.0:
            s += f"\\textcolor{{ForestGreen!{int(value*100)}!gray}}{{{value: .3g}}}"
        else:
            if convergence_flag:
                if use_colors:s += construct_cell_color("ForestGreen", 10)
                if use_symbols: s += f"\\textcolor{{ForestGreen}}{{{CHECK_SYMBOL}}}"
            else:
                if use_colors: s += construct_cell_color("gray", 10)
                if use_symbols: s += f"\\textcolor{{gray}}{{{CROSS_SYMBOL}}}"
    

    #Add the respective linker
    s +=TAB_LINKER if not final_element else TAB_END
    return s


def values_and_index_best(data: np.ndarray, order: list):
    """Function to obtain the best values of the configurations. It uses the corresponding order
    and given data
    
    # Returns:
    
    - ```best_per_conf```: np.ndarray - Best values in each configuration and variable
    - ```index_best_conf```: np.ndarray - Index where are finded such value best values
    - ```best_global```: np.ndarray - Best values per variable across all the configurations
    - ```index_best_global```: np.ndarray - Index of the best global results
    """
    # Where to save the best values and index 
    max_per_variable = []
    index_maxPV = []
    global_val = []
    index_global_val = []

    #Iteration to get the values
    for v in order:
        variable = data[:, v, :]
        evaluator = np.max if v==3 else np.min
        evaluator_idx = np.argmax if v==3 else np.argmin
        max_values = []
        index = []
        for (c, result) in enumerate(variable):
            max_values.append(evaluator(result).item())
            index.append([c, evaluator_idx(result).item(), v])
        max_per_variable.append(max_values)
        index_maxPV.append(index)
        global_val.append(evaluator(max_values).item())
        index_global_val.append(index[evaluator_idx(max_values).item()])

    #Convert into numpy arrays
    max_per_variable = np.array(max_per_variable).reshape(-1, 3).tolist()
    index_maxPV = np.array(index_maxPV).reshape(-1, 3).tolist()
    global_val = np.array(global_val).reshape(-1).tolist()
    index_global_val = index_global_val

    return max_per_variable, index_maxPV, global_val, index_global_val

## Print of each of the informative tables.


In [ ]:

# ------------------------------- CONFIGURATION -------------------------------

configurations = ["Simplest", "Original", "With LS", "Standar"]
stablished_order = {0: "Iterations", 1: "Time", 2: "LastGrad", 3: "It/sec", 4: "Convergence"}
list_order = stablished_order.keys() #Original order
list_order = [4, 0, 1, 2, 3] # First convergence
USE_COLORS = False
problem_to_print = list(range(2, result_tensor_simplest.shape[1]))


for problem in problem_to_print:
#for problem in range(result_tensor_simplest.shape[1]):
    HEADER  =  f"""
\\begin{{tabular}}{{ccccccc}} 
    \\toprule
    \\multicolumn{{7}}{{c}}{{Function Name: {names_frames[problem]} (Dim: {problem_dimensions[names_frames[problem]]})}} \\\\
    \\midrule
    Config & Algoritmo & C &Iters & Time (s) & Last Norm & It/sec\\\\
    \\midrule
"""
    print(HEADER, end="")
    #Get the result table for the current problem
    info = np.array([result_tensor_simplest[:,problem,:], 
            result_tensor_original[:,problem,:],
            result_tensor_addedLS[:,problem,:]])
    
    #Get the index of the best local and global values
    best_values_packages = values_and_index_best(info, list_order)
    index_local, index_global = best_values_packages[1], best_values_packages[-1]

    #Print configurations 
    for k in range(len(info)):

        #Moviment over the methods
        for j in range(info[k].shape[1]):

            #If the method is [GDLS, BFGS, then do not print]
            if j in [0,4]: pass

            #Otherwise
            else:
                #The method coverged? (flag used to avoid HL no convergent values)
                convergence_flag_method = bool(info[k, -1, j])
                s: str = ""
                if j==1:
                   extra_element = f"\\multirow{{4}}{{*}}{{{configurations[k]}}}"
                else:
                    extra_element = ""                 
                print(extra_element + TAB_LINKER + col_names[j], end=TAB_LINKER)

                #Moviment over the variables
                for i in list_order:
                    
                    #IDENTIFIER INSIDE THE TABLE
                    local_id = [k, j, i]
                    best_local_flag = local_id in index_local
                    best_global_flag = local_id in index_global

                    #Get the value in the adecuate format
                    if i == list_order[-1]:
                        print(format_column(info[k, i, j], j, i, use_symbols=True, use_colors=USE_COLORS, final_element=True,
                                            is_local_best=best_local_flag, is_global_best=best_global_flag, 
                                            it_converged=convergence_flag_method))
                    else:
                        print(format_column(info[k, i, j], j, i, use_symbols=True, 
                                            use_colors=USE_COLORS, is_local_best=best_local_flag, is_global_best=best_global_flag,
                                            it_converged=convergence_flag_method), end="")
        print("\\cline{2-7}")
        
    #Print methods that are equal in several executions
    for j in range(info[0].shape[1]):
        if j not in [0,4]: pass
        else:
            if j==0:
                extra_element = f"\\multirow{{2}}{{*}}{{{configurations[-1]}}}"
            else:
                extra_element = ""                 
            print(extra_element + TAB_LINKER + col_names[j], end=TAB_LINKER)
            
            #Moviment over the variables
            for i in list_order:

                #ID INSIDE THE TABLE
                local_id = [k, j, i]
                best_local_flag = local_id in index_local
                best_global_flag = local_id in index_global

                #Format of the element in the table
                if i == list_order[-1]:
                    print(format_column(info[k, i, j], j, i, use_symbols=True, use_colors=USE_COLORS, final_element=True,
                                is_local_best=best_local_flag, is_global_best=best_global_flag, it_converged=convergence_flag_method))
                else:
                    print(format_column(info[k, i, j], j, i, use_symbols=True, 
                                use_colors=USE_COLORS, is_local_best=best_local_flag, is_global_best=best_global_flag,
                                it_converged=convergence_flag_method), end="")
    
    END_HEADER = "\\bottomrule\n\\end{tabular}"
    print(END_HEADER, end="") 
    print("\n", end=TAB_LINKER) if problem % 2 == 0 else print("\n", end=TAB_END+TAB_END)


\begin{tabular}{ccccccc} 
    \toprule
    \multicolumn{7}{c}{Function Name: BEALE (Dim: 2)} \\
    \midrule
    Config & Algoritmo & C &Iters & Time (s) & Last Norm & It/sec\\
    \midrule
\multirow{4}{*}{Simplest} & Oviedo & \textcolor{ForestGreen}{\checkmark} &  33 &  0.3741 & $1.913 \cdot 10^{-14}$ &  88.20\\
 & Grads & \textcolor{ForestGreen}{\checkmark} &  59 &  0.09984 & $5.337 \cdot 10^{-17}$ &  590.93\\
 & Random & \textcolor{ForestGreen}{\checkmark} & \textcolor{ForestGreen}{\textbf{ 2}} &  3.576e-05 & $1.473 \cdot 10^{-13}$ & \textbf{ 580398.59}\\
 & Newton & \textcolor{ForestGreen}{\checkmark} &  2 & \textcolor{ForestGreen}{\textbf{ 2.694e-05}} & 0 &  74235.47\\
\cline{2-7}
\multirow{4}{*}{Original} & Oviedo & \textcolor{gray}{\texttimes} &  1000 &  0.3986 & \textcolor{BrickRed}{\texttimes} &  2508.96\\
 & Grads & \textcolor{gray}{\texttimes} &  1000 &  0.6501 & $6.848 \cdot 10^{7}$ &  1538.33\\
 & Random & \textcolor{gray}{\texttimes} &  1000 &  0.003945 & $9.883 \cdot 10

In [40]:
print(HEADER)
print(END_HEADER)


\begin{tabular}{ccccccc} 
    \toprule
    \multicolumn{7}{c}{Function Name: BRKMCC (Dim: 2)} \\
    \midrule
    Config & Algoritmo & C &Iters & Time (s) & Last Norm & It/sec\\
    \midrule

\bottomrule
\end{tabular}
